### How many taxi trips will occur in each zone, each hour?


In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import matplotlib
matplotlib.use("Agg")
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip
import logging, os, warnings
warnings.filterwarnings("ignore")

In [2]:
builder = (
    SparkSession.builder
        .appName("ForecastDemand")

        # Delta Lake
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

        # Memory (CRITICAL for 100M+ rows)
        .config("spark.driver.memory", "8g")
        .config("spark.executor.memory", "8g")

        # CPU cores
        .config("spark.driver.cores", "4")

        # Shuffle tuning
        .config("spark.sql.shuffle.partitions", "200")

        # Adaptive query optimization
        .config("spark.sql.adaptive.enabled", "true")

        # Prevent huge driver result crashes
        .config("spark.driver.maxResultSize", "2g")

        #UI tuning
        .config("spark.ui.showConsoleProgress", "false")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()


In [3]:
log_dir = "../../logs"
os.makedirs(log_dir, exist_ok=True)
logging.basicConfig(
    filename=os.path.join(log_dir, "forecast_demand_model_training.log"),
    level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s"
)

In [4]:
df = (
    spark.read.format("delta")
    .load(r"..\..\04_storage_platinum\demand_series")
    .toPandas()
)

In [5]:
# Build proper datetime column for sorting
df["ds"] = (
    pd.to_datetime(df["trip_date"]) +
    pd.to_timedelta(df["pickup_hour"], unit="h")
)
df = df.sort_values(["pu_location_id", "ds"]).reset_index(drop=True)

logging.info(f"Loaded demand_series: {len(df)} rows, {df['pu_location_id'].nunique()} zones")


In [6]:
# --------------------------------------------------
# TRAIN / TEST SPLIT
# Last 30 days = test (holdout); everything else = train
# This simulates a real deployment: model trained on
# historical data, evaluated on unseen future dates.
# --------------------------------------------------

CUTOFF     = df["ds"].max() - pd.Timedelta(days=30)
train_df   = df[df["ds"] <= CUTOFF].dropna(subset=["lag_1h", "lag_24h", "lag_168h"])
test_df    = df[df["ds"] >  CUTOFF].dropna(subset=["lag_1h", "lag_24h", "lag_168h"])

logging.info(f"Train: {len(train_df)} rows | Test: {len(test_df)} rows")

In [7]:
# --------------------------------------------------
# FEATURES
# zone_id is passed as categorical — LightGBM learns
# per-zone demand level without one-hot encoding,
# which would explode dimensionality at 265 zones.
# --------------------------------------------------

FEATURE_COLS = [
    "pu_location_id",      # categorical — zone identity
    "pickup_hour",
    "day_of_week",
    "month",
    "trip_year",
    "is_weekend",
    "is_rush_hour",
    "is_night",
    "lag_1h",
    "lag_24h",
    "lag_168h",
    "avg_fare",
    "avg_distance",
    "avg_passengers",
]
TARGET = "trip_count"

X_train = train_df[FEATURE_COLS]
y_train = train_df[TARGET]
X_test  = test_df[FEATURE_COLS]
y_test  = test_df[TARGET]

### LIGHTGBM — GLOBAL MODEL
##### One model, all zones. pu_location_id is categorical.

In [8]:
lgb_train = lgb.Dataset(
    X_train, label=y_train,
    categorical_feature=["pu_location_id"],
    free_raw_data=False
)
lgb_val = lgb.Dataset(
    X_test, label=y_test,
    categorical_feature=["pu_location_id"],
    reference=lgb_train,
    free_raw_data=False
)

In [9]:
params = {
    "objective":        "regression_l1",   # MAE loss — robust to demand spikes
    "metric":           ["mae", "rmse"],
    "learning_rate":    0.05,
    "num_leaves":       127,               # 2^7 — balanced depth for tabular data
    "min_child_samples": 20,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq":     5,
    "reg_alpha":        0.1,
    "reg_lambda":       1.0,
    "verbose":          -1,
    "n_jobs":           -1,
    "seed":             42,
}

In [10]:
callbacks = [
    lgb.early_stopping(stopping_rounds=50, verbose=True),
    lgb.log_evaluation(period=100)
]

In [11]:
logging.info("Training global LightGBM model...")
model = lgb.train(
    params,
    lgb_train,
    num_boost_round=1000,
    valid_sets=[lgb_train, lgb_val],
    valid_names=["train", "val"],
    callbacks=callbacks
)
logging.info(f"Best iteration: {model.best_iteration}")

Training until validation scores don't improve for 50 rounds
[100]	train's l1: 6.93949	train's rmse: 15.8565	val's l1: 7.34178	val's rmse: 17.3122
[200]	train's l1: 6.42116	train's rmse: 14.0645	val's l1: 6.81504	val's rmse: 15.146
[300]	train's l1: 6.27131	train's rmse: 13.6439	val's l1: 6.67551	val's rmse: 14.7277
[400]	train's l1: 6.17956	train's rmse: 13.3899	val's l1: 6.59606	val's rmse: 14.4763
[500]	train's l1: 6.11883	train's rmse: 13.2404	val's l1: 6.54513	val's rmse: 14.3447
[600]	train's l1: 6.08115	train's rmse: 13.1443	val's l1: 6.51556	val's rmse: 14.2657
[700]	train's l1: 6.03557	train's rmse: 13.0356	val's l1: 6.48193	val's rmse: 14.1854
[800]	train's l1: 6.00359	train's rmse: 12.957	val's l1: 6.45623	val's rmse: 14.1218
[900]	train's l1: 5.96988	train's rmse: 12.8776	val's l1: 6.43066	val's rmse: 14.0507
[1000]	train's l1: 5.94034	train's rmse: 12.8127	val's l1: 6.41084	val's rmse: 14.0057
Did not meet early stopping. Best iteration is:
[1000]	train's l1: 5.94034	train

### Evaluation

Global

In [12]:
preds = np.clip(model.predict(X_test), 0, None)

mae  = mean_absolute_error(y_test, preds)
rmse = root_mean_squared_error(y_test, preds)
r2   = r2_score(y_test, preds)
mape = np.mean(
    np.abs((y_test.values - preds) /
           np.where(y_test.values == 0, 1, y_test.values))
) * 100

print("\n" + "="*45)
print("GLOBAL MODEL — TEST SET PERFORMANCE")
print("="*45)
print(f"  MAE  : {mae:.2f} trips/hour")
print(f"  RMSE : {rmse:.2f} trips/hour")
print(f"  R²   : {r2:.4f}")
print(f"  MAPE : {mape:.2f}%")
logging.info(f"Global — MAE={mae:.2f} RMSE={rmse:.2f} R²={r2:.4f} MAPE={mape:.2f}%")


GLOBAL MODEL — TEST SET PERFORMANCE
  MAE  : 6.41 trips/hour
  RMSE : 14.01 trips/hour
  R²   : 0.9646
  MAPE : 31.92%


Demand based MAPE

In [13]:
# Split MAPE by demand level — shows model is accurate where it matters
test_df['prediction'] = preds

test_df["demand_bucket"] = pd.cut(
    test_df["trip_count"],
    bins=[0, 10, 50, 200, float("inf")],
    labels=["very low (<10)", "low (10–50)", "medium (50–200)", "high (200+)"],
    include_lowest=True
)

mape_by_bucket = (
    test_df.groupby("demand_bucket")
    .apply(lambda g: np.mean(
        np.abs((g["trip_count"] - g["prediction"]) /
               np.where(g["trip_count"] == 0, 1, g["trip_count"])) * 100
    ))
    .reset_index(name="mape")
)
print("\nMAPE by Demand Bucket:")
print(mape_by_bucket)


MAPE by Demand Bucket:
     demand_bucket       mape
0   very low (<10)  46.678586
1      low (10–50)  22.421737
2  medium (50–200)  13.253069
3      high (200+)  11.365586


Per - Zone

In [14]:
test_df = test_df.copy()
test_df["prediction"] = preds

per_zone = []
for zone_id, grp in test_df.groupby("pu_location_id"):
    y_z  = grp[TARGET].values
    p_z  = grp["prediction"].values
    per_zone.append({
        "zone_id":  zone_id,
        "mae":      round(mean_absolute_error(y_z, p_z), 2),
        "rmse":     round(root_mean_squared_error(y_z, p_z), 2),
        "r2":       round(r2_score(y_z, p_z), 4),
        "mape":     round(np.mean(np.abs((y_z - p_z) /
                         np.where(y_z == 0, 1, y_z))) * 100, 2)
    })

per_zone_df = pd.DataFrame(per_zone).sort_values("mae")
print("\nPer-zone results (sorted by MAE):")
print(per_zone_df.to_string(index=False))


Per-zone results (sorted by MAE):
 zone_id   mae  rmse     r2  mape
      10  0.80  1.33 0.7198 31.90
      62  0.95  1.52 0.6473 34.63
     197  0.95  1.58 0.6310 38.82
     193  0.98  1.52 0.6209 38.51
     216  1.02  1.55 0.6258 39.40
     260  1.07  1.71 0.6038 42.29
     223  1.16  1.84 0.5555 43.73
      91  1.18  1.88 0.7420 34.64
      71  1.21  1.98 0.6958 35.22
      95  1.24  2.05 0.5813 43.01
      82  1.33  2.11 0.5615 49.55
     168  1.35  1.96 0.5694 44.12
     179  1.38  2.19 0.6106 45.26
     264  1.38  2.08 0.6910 32.53
      72  1.40  2.23 0.6401 38.94
      66  1.42  2.40 0.7506 37.56
     177  1.42  2.22 0.6555 38.13
      36  1.52  2.62 0.8602 45.65
      35  1.53  2.38 0.6637 38.43
      39  1.62  2.47 0.7518 36.98
     130  1.64  2.25 0.4968 49.51
     243  1.65  2.37 0.6559 41.41
      65  1.67  2.52 0.6753 36.27
     129  1.69  2.47 0.5533 49.30
      49  1.69  2.49 0.6659 43.16
     146  1.74  2.88 0.5903 45.12
     152  1.80  2.57 0.6929 38.14
      33  1.8

Feature Importance

In [15]:
importance_df = pd.DataFrame({
    "feature":    model.feature_name(),
    "importance": model.feature_importance(importance_type="gain")
}).sort_values("importance", ascending=False)

print("\nFeature importances (gain):")
print(importance_df.to_string(index=False))
logging.info("Feature importances:\n" + importance_df.to_string())


Feature importances (gain):
       feature   importance
        lag_1h 1.201294e+07
      avg_fare 1.180613e+07
      lag_168h 9.611149e+06
       lag_24h 3.252329e+06
  avg_distance 2.727791e+06
pu_location_id 1.574261e+06
avg_passengers 1.509432e+06
   day_of_week 1.111639e+06
   pickup_hour 9.926969e+05
      is_night 5.400689e+05
    is_weekend 1.859546e+05
         month 1.017664e+05
  is_rush_hour 7.793280e+04
     trip_year 1.560666e+04


### Save model evaluation metrics to feed model evaluation dashboard

In [16]:
# Scored predictions
(
    spark.createDataFrame(test_df[
        ["pu_location_id", "ds", "trip_count", "prediction",
         "is_rush_hour", "is_weekend", "pickup_hour"]
    ].rename(columns={"ds": "datetime"}))
    .write.format("delta")
    .mode("overwrite")
    .option("mergeSchema", "true")
    .save(r"..\forecasting_demand\model_evaluation_data\demand_predictions")
)

# Per-zone metrics
(
    spark.createDataFrame(per_zone_df)
    .write.format("delta")
    .mode("overwrite")
    .save(r"..\forecasting_demand\model_evaluation_data\forecast_metrics")
)

# Feature importances
(
    spark.createDataFrame(importance_df)
    .write.format("delta")
    .mode("overwrite")
    .save(r"..\forecasting_demand\model_evaluation_data\forecast_feature_importance")
)

logging.info("All platinum outputs written. Forecast complete.")

### Save model for future inference (e.g. batch scoring, real-time API)

In [17]:
import joblib

model_package = {
    "model": model,
    "features": FEATURE_COLS
}

BASE_DIR = os.path.abspath("") 

# Simplified path mapping based on the notebook's location
model_dir = os.path.join(BASE_DIR, "model")
os.makedirs(model_dir, exist_ok=True) # Ensure directory exists
model_path = os.path.join(model_dir, "demand_forecast_lgbm.pkl")

joblib.dump(model_package, model_path)

logging.info(f"Model saved to {model_path}")
print(f"\nModel saved to {model_path}")


Model saved to c:\Users\gauth\Desktop\extended-medallion-architecture\products\forecasting_demand\model\demand_forecast_lgbm.pkl
